# Notebook 02 — Model Prototyping (Dry-Run)

In [1]:
from pathlib import Path
import os, sys

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (repo / "src").exists():
    repo = Path.cwd().parent.parent
sys.path.insert(0, str(repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.dataset import ensure_loaded
from src.data_pipeline.ingestion import generate_synthetic_transactions
from src.data_pipeline.graph_builder import build_pyg_data
from src.models.gatv2 import GATv2Net
from src.models.loss import AdaptiveFocalLoss
from src.model import GATv2, GATv2AMLModel
from src.utils.legacy import FocalLoss, compute_metrics


## Dry-run: forward pass on a tiny graph

In [2]:
df = generate_synthetic_transactions(n_accounts=40, n_transactions=120, seed=0)
data, _ = build_pyg_data(df)
print(f"nodes={data.num_nodes} edges={data.num_edges} features={data.num_node_features}")


[2026-08-27 20:54:42,086] [INFO] [src.data_pipeline.ingestion] Generated 120 synthetic transactions over 40 accounts (2.00% flagged)


[2026-08-27 20:54:42,112] [INFO] [src.data_pipeline.graph_builder] Built graph: 40 nodes, 120 edges, 9 node features, 2 edge features


nodes=40 edges=120 features=9


## Build the modular model

In [3]:
model = GATv2Net(
    in_channels=data.num_node_features,
    hidden_channels=48,
    heads=4,
    edge_dim=data.edge_attr.shape[-1],
    num_classes=2,
)
loss_fn = AdaptiveFocalLoss(init_alpha=0.25, init_gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3)
(loss_fn, optimizer)  # interactive visibility


(AdaptiveFocalLoss(),
 AdamW (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     decoupled_weight_decay: True
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.005
     maximize: False
     weight_decay: 0.01
 ))

## 1–2 epoch training loop (dry run)

In [4]:
import tqdm  # noqa: F401 -- optional
model.train()
for epoch in range(2):
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, edge_attr=data.edge_attr)
    mask = torch.ones(data.num_nodes, dtype=torch.bool)
    loss = loss_fn(logits[mask], data.y[mask])
    loss.backward()
    optimizer.step()
    print(f"epoch {epoch}: loss={loss.item():.4f}")


epoch 0: loss=0.1049
epoch 1: loss=0.0059


## Legacy symbol compatibility check

In [5]:
legacy = GATv2(in_channels=data.num_node_features, hidden_channels=48,
               out_channels=2, edge_dim=data.edge_attr.shape[-1])
with torch.no_grad():
    logits = legacy(data.x, data.edge_index, edge_attr=data.edge_attr)
print("legacy GATv2 forward:", logits.shape, "==", GATv2AMLModel is GATv2)


legacy GATv2 forward: torch.Size([40, 2]) == True


## Adaptive Focal Loss sanity

In [6]:
fl = FocalLoss(alpha=0.25, gamma=2.0)
l = fl(logits, data.y)
print(f"legacy FocalLoss = {l.item():.4f}")
loss_fn.update_history(0.2, 0.1)
print("alpha now:", float(loss_fn.alpha.clone()))


legacy FocalLoss = 0.1539
alpha now: 0.2549999952316284
